<a href="https://colab.research.google.com/github/belokonr/Minimum-Wage-Project/blob/main/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats

url = 'https://raw.githubusercontent.com/belokonr/Minimum-Wage-Project/refs/heads/main/data/processed/njpa_cleaned.csv'
df = pd.read_csv(url)

In [2]:
def table2_row(var, label, nj_data, pa_data):
    """Compute means, SEs, and t-stat for NJ vs PA comparison."""
    nj = nj_data[var].dropna()
    pa = pa_data[var].dropna()
    nj_mean, nj_se = nj.mean(), nj.sem()
    pa_mean, pa_se = pa.mean(), pa.sem()
    t_stat, _ = stats.ttest_ind(nj, pa, equal_var=False)
    return {
        'Variable': label,
        'NJ Mean': nj_mean, 'NJ SE': nj_se, 'NJ N': len(nj),
        'PA Mean': pa_mean, 'PA SE': pa_se, 'PA N': len(pa),
        't-stat': t_stat
    }

nj = df[df['NJ'] == 1]
pa = df[df['NJ'] == 0]


print("TABLE 2: MEANS OF KEY VARIABLES")

# Distribution of Store Types
print("\n1. Distribution of Store Types (percentages):")
for chain, name in [(1, 'Burger King'), (2, 'KFC'), (3, 'Roy Rogers'), (4, "Wendy's")]:
    nj_pct = (nj['CHAIN'] == chain).mean() * 100
    pa_pct = (pa['CHAIN'] == chain).mean() * 100
    print(f"   {name:15s}  NJ: {nj_pct:5.1f}%   PA: {pa_pct:5.1f}%")
nj_co = nj['CO_OWNED'].mean() * 100
pa_co = pa['CO_OWNED'].mean() * 100
print(f"   {'Company-owned':15s}  NJ: {nj_co:5.1f}%   PA: {pa_co:5.1f}%")

# Means in Wave 1
print("\n2. Means in Wave 1:")
wave1_vars = [
    ('EMPTOT', 'FTE employment'),
    ('FRACFT', 'Pct full-time (x100)'),
    ('WAGE_ST', 'Starting wage'),
    ('ATMIN', 'Wage=$4.25 (pct, x100)'),
    ('PMEAL', 'Price of full meal'),
    ('HRSOPEN', 'Hours open (weekday)'),
    ('BONUS', 'Recruiting bonus (pct, x100)'),
]
for var, label in wave1_vars:
    r = table2_row(var, label, nj, pa)
    scale = 100 if 'pct' in label.lower() or 'x100' in label else 1
    print(f"   {label:30s}  NJ: {r['NJ Mean']*scale:6.1f} ({r['NJ SE']*scale:.1f})"
          f"   PA: {r['PA Mean']*scale:6.1f} ({r['PA SE']*scale:.1f})"
          f"   t={r['t-stat']:5.1f}")

# Means in Wave 2
print("\n3. Means in Wave 2:")
wave2_vars = [
    ('EMPTOT2', 'FTE employment'),
    ('FRACFT2', 'Pct full-time (x100)'),
    ('WAGE_ST2', 'Starting wage'),
    ('NEWMIN', 'Wage=$5.05 (pct, x100)'),
    ('PMEAL2', 'Price of full meal'),
    ('HRSOPEN2', 'Hours open (weekday)'),
    ('SPECIAL2', 'Recruiting bonus (pct, x100)'),
]
for var, label in wave2_vars:
    r = table2_row(var, label, nj, pa)
    scale = 100 if 'pct' in label.lower() or 'x100' in label else 1
    print(f"   {label:30s}  NJ: {r['NJ Mean']*scale:6.1f} ({r['NJ SE']*scale:.1f})"
          f"   PA: {r['PA Mean']*scale:6.1f} ({r['PA SE']*scale:.1f})"
          f"   t={r['t-stat']:5.1f}")



TABLE 2: MEANS OF KEY VARIABLES

1. Distribution of Store Types (percentages):
   Burger King      NJ:  41.1%   PA:  44.3%
   KFC              NJ:  20.5%   PA:  15.2%
   Roy Rogers       NJ:  24.8%   PA:  21.5%
   Wendy's          NJ:  13.6%   PA:  19.0%
   Company-owned    NJ:  34.1%   PA:  35.4%

2. Means in Wave 1:
   FTE employment                  NJ:   20.4 (0.5)   PA:   23.3 (1.4)   t= -2.0
   Pct full-time (x100)            NJ:   32.8 (1.3)   PA:   35.0 (2.7)   t= -0.7
   Starting wage                   NJ:    4.6 (0.0)   PA:    4.6 (0.0)   t= -0.4
   Wage=$4.25 (pct, x100)          NJ:   30.5 (2.5)   PA:   32.9 (5.3)   t= -0.4
   Price of full meal              NJ:    3.4 (0.0)   PA:    3.0 (0.1)   t=  4.0
   Hours open (weekday)            NJ:   14.4 (0.2)   PA:   14.5 (0.3)   t= -0.3
   Recruiting bonus (pct, x100)    NJ:   23.6 (2.3)   PA:   29.1 (5.1)   t= -1.0

3. Means in Wave 2:
   FTE employment                  NJ:   21.0 (0.5)   PA:   21.2 (0.9)   t= -0.1
   Pct full

In [3]:
print("Manual Simple Difference")

mean_diff = (nj['EMPTOT2'].mean()-nj['EMPTOT'].mean()) - (pa['EMPTOT2'].mean()-pa['EMPTOT'].mean())

print("The causal estimate from simple difference is that the minimum wage increase employment by\n",
      mean_diff, "FTE workers per store relative to PA.")


Manual Simple Difference
The causal estimate from simple difference is that the minimum wage increase employment by
 2.753605782980582 FTE workers per store relative to PA.


In [4]:
print("Regression Replication")

# Restrict to stores with valid employment and wage data in both waves
reg_df = df.dropna(subset=['DEMP', 'WAGE_ST', 'WAGE_ST2', 'EMPTOT', 'EMPTOT2']).copy()

print(f"\nRegression sample size: {len(reg_df)}")
print(f"Mean of dep var (DEMP): {reg_df['DEMP'].mean():.3f}")
print(f"SD of dep var (DEMP):   {reg_df['DEMP'].std():.3f}")

# Column (i): NJ dummy only
print("\nColumn (i): NJ dummy, no controls")
m1 = smf.ols('DEMP ~ NJ', data=reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['SHEET']})
print(f"   NJ coefficient: {m1.params['NJ']:.2f} ({m1.bse['NJ']:.2f})")
print(f"   Residual SE: {np.sqrt(m1.mse_resid):.2f}")


# Column (ii): NJ dummy + chain/ownership controls
print("\nColumn (ii): NJ dummy + chain & ownership controls")
m2 = smf.ols('DEMP ~ NJ + BK + KFC + ROYS + CO_OWNED', data=reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['SHEET']})
print(f"   NJ coefficient: {m2.params['NJ']:.2f} ({m2.bse['NJ']:.2f})")
print(f"   Residual SE: {np.sqrt(m2.mse_resid):.2f}")
# F-test for controls
f_test = m2.f_test('BK = KFC = ROYS = CO_OWNED = 0')
print(f"   P-value for controls: {f_test.pvalue:.2f}")

# Column (iii): GAP only
print("\nColumn (iii): GAP variable, no controls ")
m3 = smf.ols('DEMP ~ GAP', data=reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['SHEET']})
print(f"   GAP coefficient: {m3.params['GAP']:.2f} ({m3.bse['GAP']:.2f})")
print(f"   Residual SE: {np.sqrt(m3.mse_resid):.2f}")

# Column (iv): GAP + chain/ownership
print("\n Column (iv): GAP + chain & ownership controls ")
m4 = smf.ols('DEMP ~ GAP + BK + KFC + ROYS + CO_OWNED', data=reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['SHEET']})
print(f"   GAP coefficient: {m4.params['GAP']:.2f} ({m4.bse['GAP']:.2f})")
print(f"   Residual SE: {np.sqrt(m4.mse_resid):.2f}")
f_test4 = m4.f_test('BK = KFC = ROYS = CO_OWNED = 0')
print(f"   P-value for controls: {f_test4.pvalue:.2f}")

# Column (v): GAP + chain/ownership + region
print("\n Column (v): GAP + chain/ownership + region controls ")
m5 = smf.ols('DEMP ~ GAP + BK + KFC + ROYS + CO_OWNED + CENTRALJ + SOUTHJ + PA1 + PA2',
             data=reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': reg_df['SHEET']})
print(f"   GAP coefficient: {m5.params['GAP']:.2f} ({m5.bse['GAP']:.2f})")
print(f"   Residual SE: {np.sqrt(m5.mse_resid):.2f}")
f_test5 = m5.f_test('BK = KFC = ROYS = CO_OWNED = CENTRALJ = SOUTHJ = PA1 = PA2 = 0')
print(f"   P-value for all controls: {f_test5.pvalue:.2f}")




Regression Replication

Regression sample size: 351
Mean of dep var (DEMP): -0.030
SD of dep var (DEMP):   8.748

Column (i): NJ dummy, no controls
   NJ coefficient: 2.28 (1.45)
   Residual SE: 8.71

Column (ii): NJ dummy + chain & ownership controls
   NJ coefficient: 2.28 (1.45)
   Residual SE: 8.72
   P-value for controls: 0.25

Column (iii): GAP variable, no controls 
   GAP coefficient: 17.05 (6.15)
   Residual SE: 8.66

 Column (iv): GAP + chain & ownership controls 
   GAP coefficient: 16.36 (6.54)
   Residual SE: 8.68
   P-value for controls: 0.36

 Column (v): GAP + chain/ownership + region controls 
   GAP coefficient: 13.88 (7.06)
   Residual SE: 8.67
   P-value for all controls: 0.35


In [5]:
results_df = pd.DataFrame({
    '(i)': m1.params,
    '(ii))': m2.params,
    '(iii))': m3.params,
    '(iv))':m4.params,
    '(v)': m5.params,

})

print("Regression Results Table:\n", results_df.to_markdown())


Regression Results Table:
 |           |       (i) |      (ii)) |    (iii)) |      (iv)) |        (v) |
|:----------|----------:|-----------:|----------:|-----------:|-----------:|
| BK        | nan       |   0.756644 | nan       |   0.170573 |  -0.116601 |
| CENTRALJ  | nan       | nan        | nan       | nan        |  -1.32695  |
| CO_OWNED  | nan       |   0.372914 | nan       |   0.510367 |   0.219247 |
| GAP       | nan       | nan        |  17.0516  |  16.3631   |  13.879    |
| Intercept |  -1.87879 |  -2.20668  |  -1.47773 |  -1.40261  |  -0.394124 |
| KFC       | nan       |   0.991195 | nan       |   0.564335 |   0.28016  |
| NJ        |   2.27686 |   2.28151  | nan       | nan        | nan        |
| PA1       | nan       | nan        | nan       | nan        |  -3.48175  |
| PA2       | nan       | nan        | nan       | nan        |   0.667762 |
| ROYS      | nan       |  -1.32804  | nan       |  -1.55896  |  -2.02929  |
| SOUTHJ    | nan       | nan        | nan       